In [19]:
#---Avocado---

import pandas as pd

#---STEP 2---

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# Load the dataset
df = pd.read_csv('avocado.csv')

# Drop irrelevant index column, check for NaN
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"NAN present: ", df.isna().values.any(),'\n')

#---STEP 3---

df = df.drop(columns=['Date', 'region'])

print(df.head(10))

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Example DataFrame setup (Replace with your actual column names)
# df = pd.read_csv('your_dataset.csv').dropna()

# 1. Separate features into numerical and categorical lists
num_features = ['AveragePrice', 'Total Volume', '4046', '4225', '4770', 'Total Bags',
               'Small Bags', 'Large Bags', 'XLarge Bags', 'year']

cat_features = ['type']

# using ColumnTransformer
# scale numbers and encode categories simultaneously
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ],
    remainder='drop' # Drops any columns not specified in the lists above
)

# Fit and transform the features
X = df[num_features + cat_features]
X_processed = preprocessor.fit_transform(X)

# Convert back into a readable DataFrame with new column names
encoded_cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)
all_feature_names = num_features + list(encoded_cat_names)

df_processed = pd.DataFrame(X_processed, columns=all_feature_names)

#---STEP 5---

from sklearn.model_selection import train_test_split

y = df['AveragePrice'].values

# Split off the training set (80%) and create a temporary set (20%)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_processed, y, 
    test_size=0.20, 
    random_state=42
)

# Split the remaining 20% in half (50/50)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50, 
    random_state=42
)

# Verify the final proportions
print(f"Train size: {len(X_train)} rows")
print(f"Validation size: {len(X_val)} rows")
print(f"Test size: {len(X_test)} rows")

#---STEP 6---

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

# Initialize variables to keep track of the best score and best k
best_k = 1
best_val_r2 = -float('inf')
r2_scores_list = []

# Try k values from 1 to 20
k_values = range(1, 21)

# 2. Loop through different k values to validate performance
for k in k_values:
    # Initialize and train the model
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    # Predict on the validation set
    y_val_pred = knn.predict(X_val)
    
    # Calculate R-squared score on validation set
    val_r2 = r2_score(y_val, y_val_pred)
    r2_scores_list.append(val_r2)
    
    # Track the best performing k
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_k = k

print(f"Best k found on Validation Set: k = {best_k} (Validation R² = {best_val_r2:.4f})")

# Train the final model using the absolute best k
final_knn = KNeighborsRegressor(n_neighbors=best_k)
final_knn.fit(X_train, y_train)

# Evaluate and print the final R-squared score on the unseen Test Set
y_test_pred = final_knn.predict(X_test)
final_test_r2 = r2_score(y_test, y_test_pred)

print(f"Final KNN Regressor R² Score on Test Set: {final_test_r2:.4f}\n")

#---STEP 7---

# TRY RANDOM FOREST REGRESSOR

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

#Initialize variables to keep track of the best score and best depth
best_depth = None
best_val_r2 = -float('inf')

# Try different max_depth values (None = trees expand until all leaves pure)
depth_options = [3, 5, 10, 15, 20, None]

# Loop through different max_depth values to validate performance
for depth in depth_options:
    # Initialize and train the Random Forest
    # n_estimators=100 is standard; random_state ensures reproducible results
    rf = RandomForestRegressor(max_depth=depth, n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    # Predict on the validation set
    y_val_pred = rf.predict(X_val)
    
    # Calculate R-squared score on validation set
    val_r2 = r2_score(y_val, y_val_pred)
    
    # Track the best performing depth
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_depth = depth

print(f"Best max_depth found on Validation Set: {best_depth} (Validation R² = {best_val_r2:.4f})")

# Train the final Random Forest model using the absolute best max_depth
final_rf = RandomForestRegressor(max_depth=best_depth, n_estimators=100, random_state=42)
final_rf.fit(X_train, y_train)

# Evaluate and print the final R-squared score on the unseen Test Set
y_test_pred = final_rf.predict(X_test)
final_test_r2 = r2_score(y_test, y_test_pred)

print(f"Final Random Forest Regressor R² Score on Test Set: {final_test_r2:.4f}")





NAN present:  False 

   AveragePrice  Total Volume     4046       4225    4770  Total Bags  \
0          1.33      64236.62  1036.74   54454.85   48.16     8696.87   
1          1.35      54876.98   674.28   44638.81   58.33     9505.56   
2          0.93     118220.22   794.70  109149.67  130.50     8145.35   
3          1.08      78992.15  1132.00   71976.41   72.58     5811.16   
4          1.28      51039.60   941.48   43838.39   75.78     6183.95   
5          1.26      55979.78  1184.27   48067.99   43.61     6683.91   
6          0.99      83453.76  1368.92   73672.72   93.26     8318.86   
7          0.98     109428.33   703.75  101815.36   80.00     6829.22   
8          1.02      99811.42  1022.15   87315.57   85.34    11388.36   
9          1.07      74338.76   842.40   64757.44  113.00     8625.92   

   Small Bags  Large Bags  XLarge Bags          type  year  
0     8603.62       93.25          0.0  conventional  2015  
1     9408.07       97.49          0.0  conventional